<a href="https://colab.research.google.com/github/Justine-Lewis/carisurg_portfolio/blob/main/notebooks/Week7_Interim_ModelOptimisation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/Justine-Lewis/carisurg_portfolio/blob/main/notebooks/Week7_Interim_ModelOptimisation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# WEEK 7 INTERIM DELIVERABLE

## Model Optimisation & Initial Benchmarking

This interim notebook extends the Week 6 baseline work by:

- reusing the same cleaned dataset and the same 80/20 stratified train/test split;
- retraining the Week 6 logistic regression baseline for a fair timed comparison;
- training one more-complex model: a Random Forest;
- comparing the models using accuracy, precision, recall, F1, training time, inference time per prediction, ESI 1 recall, and interpretability.

**Complex model selected for the interim:** Random Forest

### Justine Lewis
*July 2026*


## 1. Week 5 cleaning pipeline and Week 6 modelling setup

The cells below reuse the same data-cleaning and modelling logic from the Week 6 notebook.  
The Week 7 tutorials require the same data and the same `random_state = 42` split so that the comparison between the baseline and the complex model is honest.


In [26]:
#importing essential libraries
import numpy as np                 # numerical helpers (NaN, medians, etc.)
import pandas as pd                # tables / DataFrames — our main tool
import matplotlib.pyplot as plt    # plotting

# Let pandas show more of a wide table when we print it:
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 140)

print("Environment ready · pandas", pd.__version__)

Environment ready · pandas 2.2.2


In [28]:
import os

print(os.path.getsize("/content/yaleemmlc_admissionprediction_triage_full.csv"))

4194304


In [36]:
from pathlib import Path

# Define project paths
DATA_PATH = Path("/content/yaleemmlc_admissionprediction_triage_full.csv")
DOCS_PATH = Path("/content/docs")
DATA_DIR = Path("/content/data")

DOCS_PATH.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)

# Load the raw Week 5 triage dataset
df = pd.read_csv(DATA_PATH)

print("File being read:", DATA_PATH)
print("Raw dataset shape:", df.shape)
print("Total missing values:", df.isna().sum().sum())
print("Columns with missing values:", (df.isna().sum() > 0).sum())

File being read: /content/yaleemmlc_admissionprediction_triage_full.csv
Raw dataset shape: (55121, 226)
Total missing values: 0
Columns with missing values: 0


In [37]:
#Defining the target, TARGET = "esi".
TARGET = "esi"

PLAUSIBLE = {
    "triage_vital_hr": (20, 250),
    "triage_vital_sbp": (50, 300),
    "triage_vital_dbp": (30, 200),
    "triage_vital_rr": (5, 80),
    "triage_vital_o2": (50, 100),
    "triage_vital_temp": (90, 110),   # Fahrenheit
    "triage_glucose": (20, 1000)
}

def classify_columns(df):
    """Group dataset columns into clinically meaningful families."""
    return {
        "vitals": [
            col for col in df.columns
            if col.startswith("triage_vital_") or col == "triage_glucose"
        ],
        "chief_complaints": [
            col for col in df.columns
            if col.startswith("cc_")
        ],
        "demographics": [
            col for col in [
                "age", "gender", "ethnicity", "race", "lang",
                "religion", "maritalstatus", "employstatus",
                "insurance_status"
            ]
            if col in df.columns
        ],
        "admin": [
            col for col in [
                "dep_name", "arrivalmode", "arrivalmonth",
                "arrivalday", "arrivalhour_bin"
            ]
            if col in df.columns
        ],
        "leakage": [
            col for col in [
                "disposition", "previousdispo"
            ]
            if col in df.columns
        ]
    }


def clean_triage(raw):
    """
    Week 5 cleaning pipeline.

    Takes the raw triage DataFrame and returns a cleaned copy
    suitable for Week 6 baseline modelling.
    """
    d = raw.copy()

    # Remove accidental index columns created during CSV export.
    unnamed_cols = [col for col in d.columns if col.startswith("Unnamed")]
    if unnamed_cols:
        d = d.drop(columns=unnamed_cols)

    fam = classify_columns(d)

    # 1. Drop rows with no triage label.
    # We cannot train a supervised model on rows with no recorded target.
    d = d[d[TARGET].notna()].copy()

    # 2. Make vital-sign columns and age numeric.
    # Any stray text becomes NaN and can then be handled consistently.
    numeric_cols = list(fam["vitals"])
    if "age" in d.columns:
        numeric_cols.append("age")

    for col in numeric_cols:
        d[col] = pd.to_numeric(d[col], errors="coerce")

    # 3. Flag physiologically implausible values as missing.
    # These are not capped because extreme invalid values should not be treated as real.
    for col, (low, high) in PLAUSIBLE.items():
        if col in d.columns:
            out_of_range = (d[col] < low) | (d[col] > high)
            d.loc[out_of_range, col] = np.nan

    # 4. Fill vital-sign gaps using the median.
    # The median is robust to extreme values.
    for col in fam["vitals"]:
        d[col] = d[col].fillna(d[col].median())

    # 5. Fill oxygen-device and chief-complaint flags with 0.
    # Blank flag fields are treated as "not recorded/present".
    if "triage_vital_o2_device" in d.columns:
        d["triage_vital_o2_device"] = d["triage_vital_o2_device"].fillna(0)

    for col in fam["chief_complaints"]:
        d[col] = d[col].fillna(0)

    # 6. Fill text categories with "Unknown".
    # This keeps rows rather than silently dropping patients with incomplete labels.
    for col in fam["demographics"] + fam["admin"] + fam["leakage"]:
        if col in d.columns and d[col].dtype == object:
            d[col] = d[col].fillna("Unknown")

    # 7. Convert target to whole-number ESI class.
    d[TARGET] = d[TARGET].round().astype(int)

    # Keep only valid ESI classes 1-5.
    d = d[d[TARGET].between(1, 5)].copy()

    return d


In [38]:
# Apply the cleaning pipeline
df_clean = clean_triage(df)

print("Raw shape:", df.shape)
print("Cleaned shape:", df_clean.shape)
print("Missing values after cleaning:", df_clean.isna().sum().sum())
print("ESI classes after cleaning:")
print(df_clean[TARGET].value_counts().sort_index())

Raw shape: (55121, 226)
Cleaned shape: (55121, 225)
Missing values after cleaning: 0
ESI classes after cleaning:
esi
1       77
2    17924
3    27010
4     8896
5     1214
Name: count, dtype: int64


In [39]:
# Save cleaned dataset for Week 6 modelling
CLEAN_DATA_PATH = Path("triage_cleaned_v1.csv")
df_clean.to_csv(CLEAN_DATA_PATH, index=False)

print("Saved cleaned dataset to:", CLEAN_DATA_PATH)

Saved cleaned dataset to: triage_cleaned_v1.csv


In [40]:
# This cell imports the modelling and evaluation tools needed for the
# Week 6 interim baseline model deliverable.

'''
scikit-learn (sklearn): the machine-learning toolkit
joblib: saves a trained model to a file so we can reuse it later
'''
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    recall_score,
)


#Initializing a constant random seed for reproducibility.
RANDOM_SEED = 42


print(" Week 6 modelling libraries loaded.")
print(" Random seed:", RANDOM_SEED)


 Week 6 modelling libraries loaded.
 Random seed: 42


In [41]:
# ------------------------------------------------------------
# TARGET AND FEATURE DEFINITIONS
# ------------------------------------------------------------
# Target: ESI triage level. Lower ESI values usually indicate higher acuity.
TARGET = "esi"

#Columns known only after triage or after the visit are excluded to avoid leakage.
LEAKAGE = ["disposition", "previousdispo"]

#Administrative/context columns are excluded for this baseline because the
#interim target is a simple clinically interpretable baseline, not a full model.
ADMIN = ["dep_name", "arrivalmode", "arrivalmonth", "arrivalday", "arrivalhour_bin"]

DEMOGRAPHICS = [
    "age", "gender", "ethnicity", "race", "lang", "religion",
    "maritalstatus", "employstatus", "insurance_status"
]


#Index-like columns should not be used as predictors.
ID_OR_INDEX = ["Unnamed: 0"]

#Drop rows with missing target values, if any.
#Use df_clean because it has already gone through the Week 5 cleaning pipeline.
model_df = df_clean.dropna(subset=[TARGET]).copy()

# Convert ESI from float to integer labels when appropriate.
# This makes the class labels display cleanly as 1, 2, 3, 4, 5.
model_df[TARGET] = model_df[TARGET].astype(int)

# Candidate predictors start as all non-target columns minus excluded groups.
excluded_cols = set([TARGET] + LEAKAGE + ADMIN + DEMOGRAPHICS + ID_OR_INDEX)
candidate_features = [col for col in model_df.columns if col not in excluded_cols]

# Keep numeric features only for this first baseline.
# This avoids one-hot encoding at the interim stage and keeps the pipeline simple.
numeric_features = model_df[candidate_features].select_dtypes(include=[np.number]).columns.tolist()

X = model_df[numeric_features].copy()
y = model_df[TARGET].copy()

# Fill any remaining numeric gaps with the training-column median later.
# The Week 5 full dataset had 0 missing values, but this keeps the model robust.


print("Rows available for modelling:", model_df.shape[0])
print("Target:", TARGET)
print("Number of numeric model features:", len(numeric_features))
print("Excluded leakage columns:", [c for c in LEAKAGE if c in df.columns])
print("First 10 model features:", numeric_features[:10])
print("ESI class counts:")
print(y.value_counts().sort_index())

Rows available for modelling: 55121
Target: esi
Number of numeric model features: 208
Excluded leakage columns: ['disposition', 'previousdispo']
First 10 model features: ['triage_vital_hr', 'triage_vital_sbp', 'triage_vital_dbp', 'triage_vital_rr', 'triage_vital_o2', 'triage_vital_o2_device', 'triage_vital_temp', 'triage_glucose', 'cc_abdominalcramping', 'cc_abdominaldistention']
ESI class counts:
esi
1       77
2    17924
3    27010
4     8896
5     1214
Name: count, dtype: int64


In [42]:
'''80% of the data is used for training, and 20% is held out for testing.
stratify=y preserves the ESI distribution in both sets.'''
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_SEED,
)


print("Training patients:", X_train.shape[0])
print("Testing patients:", X_test.shape[0])

#check that the stratified split preserved the ESI class distribution
print("Training ESI distribution:")
print(y_train.value_counts(normalize=True).sort_index().round(3))
print("\nTesting ESI distribution:")
print(y_test.value_counts(normalize=True).sort_index().round(3))

Training patients: 44096
Testing patients: 11025
Training ESI distribution:
esi
1    0.001
2    0.325
3    0.490
4    0.161
5    0.022
Name: proportion, dtype: float64

Testing ESI distribution:
esi
1    0.001
2    0.325
3    0.490
4    0.161
5    0.022
Name: proportion, dtype: float64


## 2. Week 6 baseline: timed logistic regression

The same Week 6 logistic regression is retrained here with timing added.  
Training time measures the one-off cost of fitting the model.  
Inference time is reported in milliseconds **per patient prediction**, as required in the Week 7 benchmarking tutorial.


In [43]:
import time

# Reuse the same Week 6 logistic regression baseline.
logreg_model = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_SEED,
        class_weight="balanced",
        multi_class="auto"
    ))
])

# Measure training time.
t0 = time.perf_counter()
logreg_model.fit(X_train, y_train)
logreg_train_time = time.perf_counter() - t0

# Measure inference time per prediction.
t0 = time.perf_counter()
pred_logreg = logreg_model.predict(X_test)
logreg_infer_ms = (time.perf_counter() - t0) / len(X_test) * 1000

print("Logistic regression training time (s):", round(logreg_train_time, 4))
print("Logistic regression inference time (ms/prediction):", round(logreg_infer_ms, 6))
print("Logistic regression accuracy:", round(accuracy_score(y_test, pred_logreg), 3))


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Logistic regression training time (s): 17.9126
Logistic regression inference time (ms/prediction): 0.003732
Logistic regression accuracy: 0.562


## 3. Complex model: Random Forest

The Random Forest follows the Week 7 Tutorial 2 setup:

- `n_estimators=300`
- `class_weight="balanced"` to give more attention to rare classes such as ESI 1
- `random_state=42` for reproducibility
- `n_jobs=-1` to use the available CPU cores

Unlike logistic regression, Random Forest does not require feature scaling.


In [44]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=RANDOM_SEED,
    n_jobs=-1
)

# Measure training time.
t0 = time.perf_counter()
rf_model.fit(X_train, y_train)
rf_train_time = time.perf_counter() - t0

# Measure inference time per prediction.
t0 = time.perf_counter()
pred_rf = rf_model.predict(X_test)
rf_infer_ms = (time.perf_counter() - t0) / len(X_test) * 1000

print("Random Forest training time (s):", round(rf_train_time, 4))
print("Random Forest inference time (ms/prediction):", round(rf_infer_ms, 6))
print("Random Forest accuracy:", round(accuracy_score(y_test, pred_rf), 3))


Random Forest training time (s): 60.8122
Random Forest inference time (ms/prediction): 0.129206
Random Forest accuracy: 0.651


## 4. Initial six-axis benchmark

The Week 7 benchmark requires accuracy, precision, recall, F1, training time, and inference time, with interpretability as a seventh qualitative axis.

Because this is a five-class ESI problem, macro averaging is used for precision, recall, and F1 so that each ESI class contributes equally.  
ESI 1 recall is also retained as an additional clinically important metric because it measures how often the model identifies the most urgent patients.


In [45]:
# Retrieve ESI labels.
CLASS_LABELS = sorted(y.unique())

def safe_recall_for_label(y_true, y_pred, label=1):
    """Return recall for one ESI class or NaN if that class is absent."""
    if label not in set(y_true):
        return np.nan
    return recall_score(
        y_true,
        y_pred,
        labels=[label],
        average=None,
        zero_division=0
    )[0]

def benchmark_model(
    name,
    preds,
    train_time,
    infer_ms,
    interpretability
):
    """Summarise Week 7 performance and compute-cost metrics."""
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        y_test,
        preds,
        average="macro",
        zero_division=0
    )

    return {
        "Model": name,
        "Accuracy": round(accuracy_score(y_test, preds), 3),
        "Precision (macro)": round(precision_macro, 3),
        "Recall (macro)": round(recall_macro, 3),
        "F1 (macro)": round(f1_macro, 3),
        "Recall ESI 1": round(safe_recall_for_label(y_test, preds, label=1), 3),
        "Training Time (s)": round(train_time, 4),
        "Inference Time (ms/prediction)": round(infer_ms, 6),
        "Interpretability": interpretability,
    }

benchmark_table = pd.DataFrame([
    benchmark_model(
        "Logistic Regression (baseline)",
        pred_logreg,
        logreg_train_time,
        logreg_infer_ms,
        "High"
    ),
    benchmark_model(
        "Random Forest",
        pred_rf,
        rf_train_time,
        rf_infer_ms,
        "Medium"
    )
])

benchmark_table


,Model,Accuracy,Precision (macro),Recall (macro),F1 (macro),Recall ESI 1,Training Time (s),Inference Time (ms/prediction),Interpretability
0,Logistic Regression (baseline),0.562,0.412,0.614,0.410,0.625,17.9126,0.003732,High
1,Random Forest,0.651,0.472,0.384,0.406,0.000,60.8122,0.129206,Medium


### Interpretability note

- **Logistic Regression — High:** the model is comparatively easy to explain using its coefficients.
- **Random Forest — Medium:** global feature importances can show which variables influenced the forest overall, but hundreds of trees do not provide one simple decision path for an individual patient.

This is an initial interim comparison only. The final Week 7 submission will use the completed benchmark to support the cost-benefit recommendation.


## 5. Random Forest feature importances

Feature importance provides a simple initial interpretability check for the Random Forest by showing which predictors the model relied on most overall.


In [46]:
feature_importance = (
    pd.Series(rf_model.feature_importances_, index=X_train.columns)
    .sort_values(ascending=False)
)

print("Top 10 Random Forest feature importances:")
display(feature_importance.head(10).to_frame("importance"))


Top 10 Random Forest feature importances:


,importance
triage_vital_sbp,0.106532
triage_vital_dbp,0.095971
triage_vital_hr,0.093426
triage_glucose,0.091694
triage_vital_temp,0.080756
triage_vital_o2,0.052609
triage_vital_rr,0.052457
cc_strokealert,0.042231
cc_abdominalpain,0.026916
triage_vital_o2_device,0.026486


## 6. ESI 1 failure-mode check

The most clinically serious error remains missing an ESI 1 patient.  
This table converts ESI 1 recall into a miss rate for the logistic regression baseline and Random Forest.


In [47]:
esi1_rows = int((y_test == 1).sum())

logreg_esi1_recall = safe_recall_for_label(y_test, pred_logreg, label=1)
rf_esi1_recall = safe_recall_for_label(y_test, pred_rf, label=1)

esi1_failure_summary = pd.DataFrame([
    {
        "Model": "Logistic Regression (baseline)",
        "ESI 1 Test Cases": esi1_rows,
        "ESI 1 Recall": round(logreg_esi1_recall, 3),
        "ESI 1 Miss Rate": round(1 - logreg_esi1_recall, 3),
    },
    {
        "Model": "Random Forest",
        "ESI 1 Test Cases": esi1_rows,
        "ESI 1 Recall": round(rf_esi1_recall, 3),
        "ESI 1 Miss Rate": round(1 - rf_esi1_recall, 3),
    },
])

esi1_failure_summary


,Model,ESI 1 Test Cases,ESI 1 Recall,ESI 1 Miss Rate
0,Logistic Regression (baseline),16,0.625,0.375
1,Random Forest,16,0.000,1.000


## 7. Save the draft interim benchmark table

The interim submission also requires a draft benchmark table in the `/docs` folder.  
This cell saves both a CSV version and a Markdown version that can be committed to GitHub.


In [50]:
from pathlib import Path

DOCS_PATH = Path("/content/docs")
DOCS_PATH.mkdir(exist_ok=True)

csv_path = DOCS_PATH / "week-7-draft-benchmark.csv"
md_path = DOCS_PATH / "week-7-draft-benchmark.md"

benchmark_table.to_csv(csv_path, index=False)

with open(md_path, "w", encoding="utf-8") as f:
    f.write("# Week 7 Interim Draft Benchmark\n\n")
    f.write(
        "This table compares the Week 6 logistic regression baseline with "
        "the Week 7 Random Forest complex model using the same train/test split.\n\n"
    )
    f.write(benchmark_table.to_markdown(index=False))
    f.write("\n")

print("Saved:", csv_path)
print("Saved:", md_path)


Saved: /content/docs/week-7-draft-benchmark.csv
Saved: /content/docs/week-7-draft-benchmark.md


#Draft Benchmark Analysis of Models


- Although the Random Forest was introduced as a more complex model, it achieved an ESI 1 recall of 0.000, meaning that it failed to correctly identify any of the 16 ESI 1 patients in the test set. In comparison, the logistic regression baseline achieved an ESI 1 recall of 0.625, correctly identifying 10 of the 16 ESI 1 cases. This finding highlights the importance of evaluating class-specific recall alongside overall performance metrics, particularly when the clinically most urgent class is rare.